# EDA - Clasificacion de letalidad 2026

Este notebook resume el cambio metodologico del proyecto: el objetivo principal es un clasificador binario para estimar `fatalities > 0` usando solo eventos de 2026.

In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path('..').resolve()
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'model3_embeddings_dataset.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
df['year_calc'] = df['event_date'].dt.year
df['fatality_positive'] = (pd.to_numeric(df['fatalities'], errors='coerce').fillna(0) > 0).astype(int)

df.shape

(5120, 66)

## Distribucion temporal

Comparamos el historico con 2026 para justificar por que el modelo principal se entrena sobre el regimen operativo actual.

In [2]:
period_summary = (
    df.assign(period=df['year_calc'].where(df['year_calc'].eq(2026), '2024-2025'))
    .groupby('period')
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
    )
)
period_summary['zeros'] = period_summary['rows'] - period_summary['lethal']
period_summary[['rows', 'zeros', 'lethal', 'lethal_rate', 'fatalities']]

,rows,zeros,lethal,lethal_rate,fatalities
period,,,,,
2026,1632,1256,376,0.230392,5836.0
2024-2025,3488,3303,185,0.053039,487.0


## Foco 2026

El dataset de 2026 tiene menos filas que el historico, pero contiene una proporcion letal mucho mayor y fuentes mas alineadas con el problema actual.

In [3]:
df_2026 = df[df['year_calc'].eq(2026)].copy()

source_summary = (
    df_2026.groupby('source')
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
        max_fatalities=('fatalities', 'max'),
    )
    .sort_values('rows', ascending=False)
)
source_summary

,rows,lethal,lethal_rate,fatalities,max_fatalities
source,,,,,
iranwarlive,956,246,0.257322,4792.0,357.0
gdeltcloud,676,130,0.192308,1044.0,165.0


In [4]:
monthly_summary = (
    df_2026.groupby(df_2026['event_date'].dt.to_period('M'))
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
        max_fatalities=('fatalities', 'max'),
    )
)
monthly_summary

,rows,lethal,lethal_rate,fatalities,max_fatalities
event_date,,,,,
2026-02,2,2,1.000000,205.0,165.0
2026-03,1026,236,0.230019,3629.0,200.0
2026-04,391,94,0.240409,1537.0,357.0
2026-05,213,44,0.206573,465.0,77.0


## Variables explicativas clave

In [5]:
def categorical_lethality(column, top=12):
    summary = (
        df_2026.groupby(column)
        .agg(rows=('fatality_positive', 'size'), lethal=('fatality_positive', 'sum'), lethal_rate=('fatality_positive', 'mean'))
        .sort_values('rows', ascending=False)
        .head(top)
    )
    return summary

categorical_lethality('weapon_type')

,rows,lethal,lethal_rate
weapon_type,,,
drone,409,72,0.176039
air_strike,394,146,0.370558
unknown,362,86,0.237569
missile_rocket,345,51,0.147826
explosive_ied,75,19,0.253333
interception_air_defense,41,1,0.024390
shelling_artillery,5,1,0.200000
chemical,1,0,0.000000


In [6]:
categorical_lethality('target_type')

,rows,lethal,lethal_rate
target_type,,,
unknown,622,163,0.262058
infrastructure,332,48,0.144578
military,324,50,0.154321
civilian,256,101,0.394531
nuclear,52,6,0.115385
urban,46,8,0.173913


## Lectura para modelado

- La validacion principal debe ser temporal: entrenar hasta abril y probar en mayo.
- El split aleatorio estratificado se conserva solo como diagnostico.
- La metrica central no debe ser accuracy cruda, sino ROC-AUC, Average Precision, Balanced Accuracy, Precision, Recall y F1.
- El umbral 0.5 es inicial; el siguiente paso natural es calibrarlo segun el costo de falsas alarmas y falsos negativos.